# Circuits

# Circuits

## What This Is
Circuits in neural networks are subgraphs of components (attention heads, MLP neurons) that implement specific algorithms. Mech interp research has reverse-engineered specific circuits:

- **Indirect Object Identification (IOI)**: When reading 'Mary gave the ball to John, she told _', the model must identify 'John' as the indirect object. Researchers identified a 26-head circuit responsible.
- **Docstring completion**: How the model copies function arguments into the docstring
- **Greater-than circuit**: How GPT-2 determines which year is greater

## Circuit Analysis Methodology
1. Identify the computational task (e.g., IOI)
2. Find components that matter via activation patching
3. Trace information flow between heads
4. Verify by ablating the circuit and measuring performance drop

In [ ]:
import numpy as np
from typing import Dict, List, Tuple

np.random.seed(42)

# Simplified circuit analysis: activation patching
# Identify which components contribute to a specific output

D_MODEL = 16
N_HEADS = 4
D_HEAD = 4
SEQ = 8

def softmax(x, axis=-1):
    ex = np.exp(x - x.max(axis=axis, keepdims=True))
    return ex / ex.sum(axis=axis, keepdims=True)

class CircuitAnalyzer:
    def __init__(self, heads_per_layer=2, d_model=16, seq=8):
        np.random.seed(1)
        self.d_model = d_model
        self.n_heads = heads_per_layer
        self.d_head = d_model // heads_per_layer
        self.seq = seq
        # Initialize with some structure
        self.heads = [
            {'WQ': np.random.randn(d_model, self.d_head) * 0.2,
             'WK': np.random.randn(d_model, self.d_head) * 0.2,
             'WV': np.random.randn(d_model, self.d_head) * 0.2,
             'WO': np.random.randn(self.d_head, d_model) * 0.2}
            for _ in range(heads_per_layer * 2)  # 2 layers
        ]
        self.activations = {}
    
    def forward_with_cache(self, X, ablate_heads=None):
        ablate_heads = ablate_heads or []
        x = X.copy()
        T = x.shape[0]
        
        for l in range(2):
            for h in range(self.n_heads):
                head_idx = l * self.n_heads + h
                WQ = self.heads[head_idx]['WQ']
                WK = self.heads[head_idx]['WK']
                WV = self.heads[head_idx]['WV']
                WO = self.heads[head_idx]['WO']
                
                Q = x @ WQ; K = x @ WK; V = x @ WV
                scores = Q @ K.T / np.sqrt(self.d_head)
                scores += np.triu(np.ones((T,T))*-1e9, k=1)
                attn = softmax(scores)
                head_out = attn @ V @ WO
                
                self.activations[f'head_{l}_{h}'] = head_out.copy()
                
                if (l, h) not in ablate_heads:
                    x = x + head_out
        
        return x
    
    def activation_patch(self, X_clean, X_corrupt, target_pos, target_dim):
        """For each head, measure how much patching its output changes the target."""
        # Get clean and corrupt runs
        out_clean = self.forward_with_cache(X_clean)
        target_clean = out_clean[target_pos, target_dim]
        
        out_corrupt = self.forward_with_cache(X_corrupt)
        corrupt_cache = dict(self.activations)
        target_corrupt = out_corrupt[target_pos, target_dim]
        
        importances = {}
        for l in range(2):
            for h in range(self.n_heads):
                key = f'head_{l}_{h}'
                # Patch the corrupt run with clean head output
                x = X_corrupt.copy()
                for ll in range(2):
                    for hh in range(self.n_heads):
                        hkey = f'head_{ll}_{hh}'
                        WO = self.heads[ll*self.n_heads+hh]['WO']
                        if (ll, hh) == (l, h):
                            x = x + self.activations[key]  # patch clean
                        else:
                            x = x + corrupt_cache[hkey]  # keep corrupt
                
                target_patched = x[target_pos, target_dim]
                importance = target_patched - target_corrupt
                importances[key] = importance
        
        return importances, target_clean - target_corrupt

# Create clean and corrupt inputs
W_emb = np.random.randn(20, D_MODEL) * 0.2
tokens_clean = [3, 7, 1, 5, 3, 7, 2, 8]
tokens_corrupt = [4, 6, 1, 5, 3, 7, 2, 8]  # different first tokens
X_clean = W_emb[tokens_clean]
X_corrupt = W_emb[tokens_corrupt]

analyzer = CircuitAnalyzer()
importances, total_gap = analyzer.activation_patch(X_clean, X_corrupt, target_pos=6, target_dim=0)

print('Circuit Analysis: Activation Patching Results')
print(f'Total clean-corrupt gap: {total_gap:.4f}')
print('\nHead importance (patching clean into corrupt run):')
for head_key, imp in sorted(importances.items(), key=lambda x: abs(x[1]), reverse=True):
    pct = imp / (total_gap + 1e-8) * 100
    print(f'  {head_key}: {imp:+.4f} ({pct:+.1f}% of gap)')
